# Разбор A. Как превратить расчёт концентрации в проверяемую процедуру

У нас есть структурные карточки кофеина и истрадефиллина. Теперь нужно подготовить расчётные записи для будущего опыта: какую молярную концентрацию даст заданная масса при заданном объёме? В этом уроке массы и объёмы назначены для учебного сценария. Реальный раствор мы не готовим.

Я начну с практического требования: другой исследователь должен увидеть, какие строки обработаны, какие отклонены и почему. Красивый столбец чисел без исходных единиц и журнала ошибок для передачи недостаточен. Если одна строка исчезнет, коллега должна отличить обоснованный отказ от потери данных.

Сначала разберём смысл формулы на одной записи. Затем оформим правило как функцию, проверим его на нормальных и намеренно ошибочных условиях и только после этого построим таблицу.

In [2]:
from pathlib import Path
import sys
import json
import math
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'programm/course.json').is_file())
sys.path.insert(0, str(ROOT / 'practicuum/shared'))
from course_support import compound, workspace, write_json, init_dossier
OUT = workspace('M02-L02', 'case')


## 1. Выведем преобразование до программирования

Количество вещества n связано с массой m и молярной массой M: n = m/M. Концентрация c = n/V. В карточке M выражена в g/mol, а заданные масса и объём — в mg и mL. Поэтому я явно приведу массу к g, объём к L и получу c в mol/L:

\[
c=\frac{m_{\mathrm{mg}}/1000}{M_{\mathrm{g/mol}}\,(V_{\mathrm{mL}}/1000)}.
\]

Зачем писать оба преобразования, если множители 1000 сокращаются? Чтобы сохранить смысл промежуточных величин. Для согласованной пары mg и mL сокращённая формула численно работает. Но если следующий файл задаст массу в g, а объём останется в mL, применять прежнюю запись без пересмотра единиц будет ошибкой. Формулу сокращают после анализа размерностей, а не вместо него.

До запуска сравните кофеин и истрадефиллин при одинаковых массе и объёме. У какого вещества больше молярная масса в наших карточках? Значит, какое количество вещества будет меньше? Этот прогноз проверяет направление результата независимо от программной реализации.

In [3]:
mass_mg = 1.0
volume_ml = 10.0
molar_mass = compound('istradefylline')['molecular_weight_g_mol']
mass_g = mass_mg / 1000
volume_L = volume_ml / 1000
amount_mol = mass_g / molar_mass
c_mol_L = amount_mol / volume_L
display({'mass_g': mass_g, 'volume_L': volume_L,
         'MW_g_mol': molar_mass, 'amount_mol': amount_mol, 'c_mol_L': c_mol_L})


{'mass_g': 0.001,
 'volume_L': 0.01,
 'MW_g_mol': 384.4,
 'amount_mol': 2.6014568158168575e-06,
 'c_mol_L': 0.00026014568158168577}

Теперь я читаю результат по цепочке: 1 mg превратился в 0,001 g, 10 mL — в 0,01 L; деление массы на молярную массу дало количество вещества. Концентрация относится к назначенному объёму и выбранной записи вещества. Число знаков в выводе Python отражает вычисление, а точность приготовления придётся обосновывать отдельно.

## 2. Сначала решим, какие входы функция принимает

Функция будет получать числовые значения в явно названных единицах. Я разрешаю положительные конечные массу, молярную массу и объём. Это контракт для нашей задачи подготовки ненулевого раствора: отрицательная масса не имеет принятого смысла, нулевой объём делает формулу неопределённой, NaN и бесконечность не дают пригодной расчётной записи.

Можно было бы вернуть пустое значение и продолжить. Но тогда последующая таблица могла бы скрыть причину отказа. Я выбираю исключение `ValueError`, а обработчик каждой строки сохранит её номер и объяснение. Политика отказа становится частью процедуры, а не случайным поведением деления.

In [4]:
def concentration(mass_mg, molar_mass, volume_ml):
    values = [mass_mg, molar_mass, volume_ml]
    if not all(math.isfinite(v) and v > 0 for v in values):
        raise ValueError('Масса, молярная масса и объём должны быть конечными и положительными')
    mass_g = mass_mg / 1000
    volume_L = volume_ml / 1000
    return (mass_g / molar_mass) / volume_L

assert math.isclose(concentration(1.0, molar_mass, 10.0), c_mol_L)


Это сравнение связывает функцию с уже разобранным вычислением. Одного такого теста мало: правильный ответ при нормальном входе ничего не говорит о поведении на ошибочной строке. Поэтому следующий шаг проверяет и принятие, и отказ.

## 3. Обработаем таблицу без молчаливого исчезновения строк

Первые две строки ниже — учебные условия для сравнения веществ. Две последние я намеренно делаю ошибочными: нулевой объём и отрицательная масса. Они нужны для проверки процедуры и не описывают реальные измерения.

Для каждой строки сначала извлекаем молярную массу выбранной карточки, затем вызываем одну и ту же функцию. Успех и отказ сохраняем раздельно, оставляя исходные значения в обеих таблицах. Так можно восстановить судьбу каждой записи.

In [5]:
conditions = [
    ('caffeine', 1.0, 10.0),
    ('istradefylline', 1.0, 10.0),
    ('caffeine', 1.0, 0.0),
    ('istradefylline', -1.0, 10.0),
]
valid, rejected = [], []
for row_id, (name, mass, volume) in enumerate(conditions, start=1):
    record = {'row_id': row_id, 'name': name, 'mass_mg': mass, 'volume_ml': volume,
              'MW_g_mol': compound(name)['molecular_weight_g_mol']}
    try:
        value = concentration(mass, record['MW_g_mol'], volume)
    except ValueError as error:
        rejected.append({**record, 'reason': str(error)})
    else:
        valid.append({**record, 'c_mol_L': value})
display(valid)
display(rejected)
assert len(valid) == 2 and len(rejected) == 2
assert {r['row_id'] for r in valid + rejected} == set(range(1, len(conditions) + 1))


[{'row_id': 1,
  'name': 'caffeine',
  'mass_mg': 1.0,
  'volume_ml': 10.0,
  'MW_g_mol': 194.19,
  'c_mol_L': 0.0005149595756733096},
 {'row_id': 2,
  'name': 'istradefylline',
  'mass_mg': 1.0,
  'volume_ml': 10.0,
  'MW_g_mol': 384.4,
  'c_mol_L': 0.00026014568158168577}]

[{'row_id': 3,
  'name': 'caffeine',
  'mass_mg': 1.0,
  'volume_ml': 0.0,
  'MW_g_mol': 194.19,
  'reason': 'Масса, молярная масса и объём должны быть конечными и положительными'},
 {'row_id': 4,
  'name': 'istradefylline',
  'mass_mg': -1.0,
  'volume_ml': 10.0,
  'MW_g_mol': 384.4,
  'reason': 'Масса, молярная масса и объём должны быть конечными и положительными'}]

Здесь `try/except` не прячет ошибку: он превращает ожидаемый отказ в проверяемую запись. Последние два `assert` показывают, что получено ровно два успеха и два отказа и ни один row_id не исчез. При чтении внешней таблицы понадобилась бы ещё проверка типов и преобразования строк в числа; в нашем явно заданном списке входы уже числовые.

Теперь сравните концентрации двух принятых строк со своим прогнозом. При одинаковых массе и объёме большая молярная масса даёт меньшее количество вещества и меньшую молярную концентрацию. Если бы порядок оказался обратным, я сначала проверил бы выбор карточки и перевод единиц, а затем арифметику функции.

## 4. Почему правдоподобного числа недостаточно

Рассмотрим две ошибки, которые встречаются при переносе формулы. Если забыть только перевод mg в g, результат вырастет в 1000 раз. Если забыть только перевод mL в L, он уменьшится в 1000 раз. Чтобы не запоминать это как заклинание, вычислим оба отношения на тех же исходных значениях.

In [6]:
correct = concentration(mass_mg, molar_mass, volume_ml)
mass_conversion_missing = (mass_mg / molar_mass) / (volume_ml / 1000)
volume_conversion_missing = ((mass_mg / 1000) / molar_mass) / volume_ml
unit_error_ratio = mass_conversion_missing / correct
display({'correct_mol_L': correct,
         'mass_conversion_missing_ratio': unit_error_ratio,
         'volume_conversion_missing_ratio': volume_conversion_missing / correct,
         'mg_mL_shortcut_agrees': math.isclose(mass_mg / molar_mass / volume_ml, correct)})
assert math.isclose(unit_error_ratio, 1000.0)
assert math.isclose(volume_conversion_missing / correct, 0.001)


{'correct_mol_L': 0.00026014568158168577,
 'mass_conversion_missing_ratio': 999.9999999999998,
 'volume_conversion_missing_ratio': 0.001,
 'mg_mL_shortcut_agrees': True}

Последнее поле особенно важно: сокращённое выражение для согласованных mg и mL совпадает с явным переводом. Совпадение чисел поэтому не доказывает, что автор правильно понимает единицы. Для передачи я оставляю явные обозначения входов и промежуточных величин. Коллега сможет проверить формулу и заметить изменение формата следующего файла.

## 5. Передадим процедуру вместе с результатом

Я сохраняю принятые и отклонённые строки, затем повторно читаю JSON. Эта проверка отвечает на практический вопрос: сохранили ли мы именно ту таблицу, которую только что построили? Для переноса в B дополнительно потребуется установить форму аторвастатина и смысл массы на этикетке: функция получает молярную массу как аргумент и сама её химическую правильность не устанавливает.

In [7]:
write_json(OUT / 'valid.json', valid)
write_json(OUT / 'rejected.json', rejected)
write_json(OUT / 'table.json', valid)
assert json.loads((OUT / 'valid.json').read_text(encoding='utf-8')) == valid
assert json.loads((OUT / 'rejected.json').read_text(encoding='utf-8')) == rejected
result = {'kind': 'simulation', 'rows': valid, 'rejected': rejected,
          'unit_error_ratio': unit_error_ratio,
          'decision': 'Передавать входные единицы и журнал отказов вместе с концентрациями'}
write_json(OUT / 'result.json', result)
from dossier_records import append_record
print(append_record(init_dossier(), 'M02-L02', 'A', result, ROOT,
                    inputs=['practicuum/shared/data/compounds.json']))


C:\Users\Miakhlov\Documents\GitHub\Computational chemistry_course\tmp\reasoning-full-20261120\workspace\my_research\evidence\M02-L02\20261120T105311620574Z-A-fac15cd6fffa4bc8b0ee432b9bf07e98.json


Итак, коллега получит не только концентрации, но и объяснение судьбы каждой строки. В самостоятельной работе изменятся вещество и назначенные условия. Вашей ответственностью останутся политика формы, единицы, правила отказа и обоснование того, какие записи можно передавать дальше.

# M02. Как наставник читает первичную работу

Пока следующая вычислительная задача ещё не выбрана, вернёмся к источнику: **Crevar-Sakač et al. (2013). An Improved HPLC Method with the Aid of a Chemometric Protocol: Simultaneous Determination of Atorvastatin and Its Metabolites in Plasma**. [Локальный полный текст](../../../../../shared/data/source-responses/hplc_article.xml), [DOI](https://doi.org/10.3390/molecules18032469). Откройте **Table 1 и условия эксперимента**.

Сопоставьте одну строку первичной таблицы и CSV; прочтите заголовки до чисел.

Наставник начинает с вопроса «что здесь непосредственно наблюдали?». Найдите одно такое наблюдение и его место в источнике. Затем рядом запишите трактовку авторов и ещё одно возможное объяснение. Это учебный приём чтения, а не реконструкция мыслей авторов. Если численного ряда нет, вывод нельзя усиливать придуманными точками.

Перед вычислением назовите объект, условия, величину и единицы. После вычисления сравните ответ с исходным утверждением. При расхождении сначала проверьте форму вещества, основу концентрации и преобразования таблицы; только затем обсуждайте модель. Смысл операции — получить проверяемый аргумент для следующего решения досье.

Теперь перенесите приём на [самостоятельную мини-рецензию B](../steps/03_article_review.md). Для проверки собственного вывода используйте вопрос: **Достаточно ли одинакового числа строк для доказательства правильного переноса таблицы?**